In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the lists for d and t values
d_values = [3, 4, 5]
t_values = [1, 2]

# Define the OOC data types and their corresponding file names
ooc_types = {
    'hetero': {
        'filename': 'OOC_H_values_Spline_Simulations_New_1_d{d}_t{t}.csv',
        'label': 'Hetero'
    },
    'fBm': {
        'filename': 'OOC_H_values_Spline_Simulations_New_2_d{d}_t{t}.csv',
        'label': 'fBm'
    },
    'gn': {
        'filename': 'OOC_H_values_Spline_Simulations_New_3_d{d}_t{t}.csv',
        'label': 'GN'
    }
}

def plot_control_chart(ax, ic_data, data_to_plot, d, t, title_suffix, confidence_level):
    """
    Plots the H_value control chart for a given dataset.
    
    Args:
        ax (matplotlib.axes.Axes): The subplot axis to draw on.
        ic_data (np.ndarray): The H_value data for the In-Control process.
        data_to_plot (np.ndarray): The H_value data to plot on the chart (IC or OOC).
        d (int): The d-value for the dataset.
        t (int): The t-value for the dataset.
        title_suffix (str): Additional text for the plot title (e.g., 'IC' or 'OOC').
        confidence_level (int): The confidence level for the control limits (e.g., 95 or 99).
    """
    # Calculate quantiles based on the confidence level
    if confidence_level == 95:
        lower_quantile = 0.025
        upper_quantile = 0.975
    elif confidence_level == 99:
        lower_quantile = 0.005
        upper_quantile = 0.995
    else:
        raise ValueError("Confidence level must be 95 or 99.")
    
    # Calculate control limits from IC data
    lcl = np.quantile(ic_data, lower_quantile)
    ucl = np.quantile(ic_data, upper_quantile)
    
    # Plot the H-values
    color = 'blue' if title_suffix == 'IC' else 'purple'
    label = f'{title_suffix} H-Values'
    ax.plot(np.arange(1, len(data_to_plot) + 1), data_to_plot, 'o', markersize=5, color=color, label=label)
    
    # Plot the control limits as horizontal lines
    ax.axhline(y=lcl, color='green', linestyle='--', label=f'IC LCL: {lcl:.4f}')
    ax.axhline(y=ucl, color='red', linestyle='--', label=f'IC UCL: {ucl:.4f}')
    
    # Title and labels
    #ax.set_title(f'd={d}, t={t}, {confidence_level}% CI ({title_suffix})', fontsize=14)
    ax.set_title(fr'$d={d}, \tau={t}$, {confidence_level}% CI ({title_suffix})', fontsize=14)
    ax.set_xlabel('Simulation', fontsize=12)
    ax.set_ylabel('$H$', fontsize=12)
    ax.grid(False)

    # Place the legend in a consistent location
    ax.legend(loc='lower right', fontsize=20)

# --- Plotting IC data separately for each d value ---
for d in d_values:
    print(f"\nProcessing IC Control Chart plots for d={d}...")
    
    try:
        ic_filename_t1 = f'IC_H_values_Spline_Simulations_d{d}_t1.csv'
        ic_filename_t2 = f'IC_H_values_Spline_Simulations_d{d}_t2.csv'
        
        data_ic_t1 = pd.read_csv(ic_filename_t1)
        data_ic_t2 = pd.read_csv(ic_filename_t2)
        
        # Create a figure for the IC plots (2x2 grid)
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True)
        
        # Top row: 95% CI plots
        plot_control_chart(axes[0, 0], data_ic_t1['H_value'], data_ic_t1['H_value'], d, 1, 'IC', 95)
        plot_control_chart(axes[0, 1], data_ic_t2['H_value'], data_ic_t2['H_value'], d, 2, 'IC', 95)
        
        # Bottom row: 99% CI plots
        plot_control_chart(axes[1, 0], data_ic_t1['H_value'], data_ic_t1['H_value'], d, 1, 'IC', 99)
        plot_control_chart(axes[1, 1], data_ic_t2['H_value'], data_ic_t2['H_value'], d, 2, 'IC', 99)
        
        plt.tight_layout()
        plt.savefig(f'ControlChart_IC_d{d}.png')
        plt.close(fig)
        
    except FileNotFoundError as e:
        print(f"  Error: File not found - {e}. Skipping plots for d={d}.")
        plt.close(fig)

# --- Plotting OOC data for each d and ooc_type ---
for ooc_name, ooc_info in ooc_types.items():
    print(f"\nProcessing Control Chart plots for OOC ({ooc_info['label']})...")
    
    for d in d_values:
        print(f"  Creating plot for d={d}...")
        
        try:
            # Load the IC and OOC data
            ic_filename_t1 = f'IC_H_values_Spline_Simulations_d{d}_t1.csv'
            ooc_filename_t1 = ooc_info['filename'].format(d=d, t=1)
            ic_filename_t2 = f'IC_H_values_Spline_Simulations_d{d}_t2.csv'
            ooc_filename_t2 = ooc_info['filename'].format(d=d, t=2)
            
            data_ic_t1 = pd.read_csv(ic_filename_t1)
            data_ooc_t1 = pd.read_csv(ooc_filename_t1)
            data_ic_t2 = pd.read_csv(ic_filename_t2)
            data_ooc_t2 = pd.read_csv(ooc_filename_t2)
            
            # Create a new figure for the OOC plots for this d and ooc_type
            fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True)
            
            # Top row: 95% CI plots
            plot_control_chart(axes[0, 0], data_ic_t1['H_value'], data_ooc_t1['H_value'], d, 1, 'OOC', 95)
            plot_control_chart(axes[0, 1], data_ic_t2['H_value'], data_ooc_t2['H_value'], d, 2, 'OOC', 95)
            
            # Bottom row: 99% CI plots
            plot_control_chart(axes[1, 0], data_ic_t1['H_value'], data_ooc_t1['H_value'], d, 1, 'OOC', 99)
            plot_control_chart(axes[1, 1], data_ic_t2['H_value'], data_ooc_t2['H_value'], d, 2, 'OOC', 99)
            
            plt.tight_layout()
            plt.savefig(f'ControlChart_OOC_d{d}_{ooc_name}.png')
            plt.close(fig)
            
        except FileNotFoundError as e:
            print(f"  Error: File not found - {e}. Skipping plots for d={d}.")
            plt.close(fig)

print("\nAll Control Chart plots generated and saved successfully.")


Processing IC Control Chart plots for d=3...

Processing IC Control Chart plots for d=4...

Processing IC Control Chart plots for d=5...

Processing Control Chart plots for OOC (Hetero)...
  Creating plot for d=3...
  Creating plot for d=4...
  Creating plot for d=5...

Processing Control Chart plots for OOC (fBm)...
  Creating plot for d=3...
  Creating plot for d=4...
  Creating plot for d=5...

Processing Control Chart plots for OOC (GN)...
  Creating plot for d=3...
  Creating plot for d=4...
  Creating plot for d=5...

All Control Chart plots generated and saved successfully.
